# What does this Script do?

## Imports the Relevant data from the Partners in care Looker database, cleans and combines it, then updates a SQl table where its used to maintain a PowerBI Dashboard.  

import python packages

In [11]:
import pandas as pd
import os
import json
import datetime
from datetime import datetime, date, timezone
import numpy as np
import pytz


from io import StringIO
import looker_sdk

import custom packages

In [12]:
from utils.error_handling import Task, TaskManager

# general utilities for all payment predictions
from utils.email_utils import send_task_alert
from utils.ds_logger_threaded import get_mt_ds_logger
from ds_sql_connections import GsAutomationReliefAssistanceConnection

initialize task tracker

In [13]:
# initialization of the task manager with the overall name of the task this code is supposed to do.
task_mgr = TaskManager("partners in care clarity looker veterans data ingestion")

initialize logger

In [14]:
task = Task(task_nm="Initialize log file", task_details="")
try:
    # creating any relevant dates
    now = datetime.now()
    now_str = now.strftime("%Y%m%d_%H%M%S")
    # log file info
    logging_dir = os.path.join(os.getcwd(), "logs")
    logging_filename = (
        "partners_in_care_clarity_looker_veterans_data_ingestion_{}.log".format(now_str)
    )
    logging_filename_with_path = os.path.join(logging_dir, logging_filename)

    if not os.path.exists(logging_dir):
        os.mkdir(logging_dir)

    logger = get_mt_ds_logger(
        logging_filename_with_path,
        "Partners_In_Care",
        "Data_ingestion",
        level="INFO",
        format_string="%(asctime)s %(levelname)s %(message)s",
    )
    # updating the task tracker
    task["Details"] = "Successfully created and began writing to log file"
    task["Success"] = True
    logger.info("Data ingestion for partners in care started")

except Exception as e:
    task["Details"] = str(e)
    print(e)

task_mgr.add_task(task)

initialize variables

In [15]:
task = Task(task_nm="Initialize starting global variables", task_details="")
try:
    task_mgr.check_errors()
    main_directory_path = str(os.getcwd())
    config_path = main_directory_path + "\\utils\\config_partners_in_care.json"
    looker_config_path = main_directory_path + "\\utils\\looker.ini"

    # variables used in pulling records from looker
    model = "oahu_connection_model"
    view = "client"
    limit = 100000

    # creating the config object to pull the authentication credentials later in various calls.
    with open(config_path, "r") as file:
        config_object = json.load(file)

    sql_username = config_object["Josh"]["username"]
    sql_password = config_object["Josh"]["password"]

    today = date.today()
    today_str = today.strftime("%m_%d_%Y")
    reporting_period_today_str = today.strftime("%Y-%m-%d")

    now_utc = datetime.now(timezone.utc)
    hst = pytz.timezone("Pacific/Honolulu")
    now_hst = datetime.now(hst)
    now_utc_str = now_utc.strftime("%Y/%m/%d %H:%M:%S")
    now_str_formatted = now.strftime("%Y/%m/%d %H:%M:%S")
    now_hst_str = now_hst.strftime("%Y/%m/%d %H:%M:%S")

    # updating the task tracker
    task["Details"] = "successfully initialize global variables"
    task["Success"] = True
    logger.info("Initialized starting global variables")
except Exception as e:
    task["Details"] = str(e)
    logger.error(f"Initializing global variables failed with the following error :{e}")
    print(e)

task_mgr.add_task(task)

Grab program level information 

In [16]:
task = Task(
    task_nm="Connect to database and ingest veteran data from looker",
    task_details="",
)
try:
    task_mgr.check_errors()
    # initialize sdk
    sdk = looker_sdk.init40(looker_config_path)
    queries = [
        {
            "fields": [
                "client.id",
                "client.full_name",
                "client.unique_identifier",
                "programs.name",
                "programs.project_type_code",
                "enrollments.id",
                "enrollments.start_date",
                "enrollments.end_date",
                "client_referral_status.Currently_On_Queue",
                "chronically_homeless_at_entry.is_chronic_homeless",
                "entry_screen.disabling_condition",
                "entry_screen.chronic_7_date",
            ],
            "filters": {
                "static_demographics.veteran_text": "yes",
                "entry_screen.head_of_household": "yes",
            },
            "name": "clients_and_programs_df",
        },
        {
            "fields": [
                "client.id",
                "screens.name",
                "client_assessments.assessment_date",
            ],
            "filters": {
                "static_demographics.veteran_text": "yes",
                "entry_screen.head_of_household": "yes",
                "screens.name": "Custom VI-SPDAT Families (V2), Custom VI-SPDAT Individual Adult (V2), Custom VI-SPDAT Transition Age Youth (V1)",
            },
            "name": "assessments_df",
        },
        {
            "fields": [
                "client.id",
                "client_assessments.assessment_date",
                "client_assessment_custom.c_discharge_custom",
            ],
            "filters": {
                "static_demographics.veteran_text": "yes",
                "entry_screen.head_of_household": "yes",
                "client_assessment_custom.c_discharge_custom": "-null",
            },
            "name": "discharge_df",
        },
        {
            "fields": [
                "client.id",
                "client_file_names.name",
            ],
            "filters": {
                "static_demographics.veteran_text": "yes",
                "entry_screen.head_of_household": "yes",
                "client_file_names.name": "DD 214 (Veterans Only)",
            },
            "name": "files_df",
        },
        {
            "fields": [
                "client.id",
                "referral_connection_enrollments.id",
                "referral_connection_enrollment_programs.name",
                "referrals.status",
                "referrals.id",
                "referrals.date_accepted_in_program_date",
                "referrals.created_date",
                "referrals.added_date",
                "referto_program.project_type_code",
                "referto_program.name",
            ],
            "filters": {
                "static_demographics.veteran_text": "Yes",
                "entry_screen.head_of_household": "Yes",
                "referto_program.name": "-null",
            },
            "name": "referrals_df",
        },
    ]

    # Function to generate query body
    def generate_body(model, view, query, limit):
        return {
            "model": model,
            "view": view,
            "fields": query["fields"],
            "filters": query["filters"],
            "limit": limit,
        }

    # Function to run the query
    def run_query(sdk, model, view, query, limit):
        body = generate_body(model, view, query, limit)
        result = sdk.run_inline_query(result_format="csv", body=body)
        return result

    # Function to process the results (add Reporting Period column)
    def process_result(result):
        df = pd.read_csv(StringIO(result))
        return df

    veterans_df = None
    for query in queries:
        result = run_query(sdk, model, view, query, limit)
        df = process_result(result)
        if query["name"] == "discharge_df":
            df["Client Assessments Assessment Date"] = pd.to_datetime(
                df["Client Assessments Assessment Date"]
            )

            # Sort the dataframe by assessment date in descending order
            df_sorted = df.sort_values(
                by="Client Assessments Assessment Date", ascending=False
            )

            # Drop duplicates based on client ID, keeping only the latest entry
            df_latest = df_sorted.drop_duplicates(
                subset="Clients Client ID", keep="first"
            )

            # Drop the assessment date column
            df = df_latest.drop(columns=["Client Assessments Assessment Date"])

        if veterans_df is None:
            veterans_df = df
        else:
            if query["name"] == "referrals_df":
                # Merge the referrals data with the veterans data
                df["Referral Connected Enrollments Enrollment ID"] = (
                    df["Referral Connected Enrollments Enrollment ID"]
                    .fillna(-1)
                    .astype(int)
                )  # Convert to int to match the veterans data

                pending_statuses = ["Pending", "Pending - In Process"]
                completed_referrals = df.loc[
                    df["Referrals Status"] == "Completed",
                    [
                        "Referral Connected Enrollments Enrollment ID",
                        "Referrals ID",
                        "Referrals Created Date",
                    ],
                ]

                pending_referrals = df.loc[
                    df["Referrals Status"].isin(pending_statuses),
                    ["Clients Client ID", "Referrals ID", "Referrals Created Date"],
                ]

                veterans_df = pd.merge(
                    veterans_df,
                    completed_referrals,
                    how="left",
                    left_on="Enrollments Enrollment ID",
                    right_on="Referral Connected Enrollments Enrollment ID",
                )  # Merge on the enrollment ID for completed referrals

                veterans_df = pd.merge(
                    veterans_df,
                    pending_referrals,
                    how="left",
                    on="Clients Client ID",
                    suffixes=("_completed", "_pending"),
                )  # Merge on the client ID for pending referrals

            else:
                veterans_df = pd.merge(
                    veterans_df, df, on="Clients Client ID", how="left"
                )

    veterans_df.sort_values(by="Clients Client ID", inplace=True)

    # updating the task tracker
    task["Details"] = "Successfully grabbed veterans data from looker"
    task["Success"] = True
    logger.info("Connected to database and pulled veterans data from looker")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"connecting to database and grabbing veterans data failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

KeyboardInterrupt: 

upload to database

In [7]:
task = Task(
    task_nm="uploading new table to database",
    task_details="",
)
try:
    task_mgr.check_errors()

    veterans_df["Update Time HST"] = now_hst_str
    veterans_df["Update Time CST"] = now_str_formatted
    veterans_df["Update Time UTC"] = now_utc_str

    conn = GsAutomationReliefAssistanceConnection(sql_username, sql_password, auth=True)

    conn.insert_with_progress(
        df=veterans_df,
        schema="dbo",
        tbl_nm="partners_in_care_veterans",
        dtypes={},
    )

    # updating the task tracker
    task["Details"] = "Successfully updated data in database"
    task["Success"] = True
    logger.info("Updated data in Database")

except Exception as e:
    task["Details"] = str(e)
    logger.error(f"uploading new data to database failed with teh following error:{e}")
    print(e)

task_mgr.add_task(task)

Building connection to relief-assistance.database.windows.net - gs-automation


  0%|          | 0/14434 [00:00<?, ?it/s]

20000it [04:20, 76.64it/s]                           


In [8]:
veterans_df

,Clients Client ID,Clients Client Full Name,Clients Unique Identifier,Programs Name,Programs Project Type Code,Enrollments Enrollment ID,Enrollments Project Start Date,Enrollments Project Exit Date,Client Referral Status Is client currently on queue?,Entry Screen Chronically Homeless at Project Start - Individual,...,Entry Screen Approximate Date this Episode of Homelessness Started Date,Client Assessments Name,Client Assessments Assessment Date,Client Assessment Custom Veteran Discharge Status,Clients Client File Name,Referral Connected Enrollments Enrollment ID,Referrals ID_completed,Referrals Created Date_completed,Referrals ID_pending,Referrals Created Date_pending
13252,9,Derrick Ontai,68197A11D,Coordinated Entry Program,Coordinated Entry,22645,2017-05-01,2018-05-01,No,Missing,...,NaN,NaN,NaN,Honorable,NaN,NaN,NaN,NaN,NaN,NaN
13253,9,Derrick Ontai,68197A11D,BP O'ahu Region 3 Street Outreach - HPO,Street Outreach,22647,2017-05-01,2017-07-30,No,No,...,2017-05-17,NaN,NaN,Honorable,NaN,NaN,NaN,NaN,NaN,NaN
10590,9,Derrick Ontai,68197A11D,BP GPD Clinical Treatment - HPO,Transitional Housing,51119,2018-11-01,2019-08-30,No,No,...,2018-10-25,NaN,NaN,Honorable,NaN,NaN,NaN,NaN,NaN,NaN
10591,9,Derrick Ontai,68197A11D,Coordinated Entry Program,Coordinated Entry,52111,2018-11-01,2019-08-30,No,Missing,...,NaN,NaN,NaN,Honorable,NaN,NaN,NaN,NaN,NaN,NaN
9142,9,Derrick Ontai,68197A11D,SSVF Priority 1 Rapid Re-Housing,PH – Rapid Re-Housing,66273,2019-09-13,2020-02-24,No,No,...,2018-04-04,NaN,NaN,Honorable,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,53051,Kirsten Parness,B5B1FF273,Coordinated Entry Program,Coordinated Entry,150557,2025-01-10,NaN,Yes,No,...,2025-01-10,Custom VI-SPDAT Individual Adult (V2),2025-01-10,General,DD 214 (Veterans Only),NaN,NaN,NaN,NaN,NaN
11,53052,William Bacon,B4B423496,WCC HOPTEL,Emergency Shelter – Entry Exit,150692,2025-01-10,NaN,Yes,No,...,2025-01-10,Custom VI-SPDAT Individual Adult (V2),2025-01-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,53052,William Bacon,B4B423496,Coordinated Entry Program,Coordinated Entry,150561,2025-01-10,NaN,Yes,No,...,2025-01-10,Custom VI-SPDAT Individual Adult (V2),2025-01-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,53070,Rakab Holman,392638B4A,Coordinated Entry Program,Coordinated Entry,150649,2025-01-14,NaN,Yes,No,...,2024-11-01,Custom VI-SPDAT Individual Adult (V2),2025-01-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
veterans_df["Enrollments Project Start Date"] = pd.to_datetime(
    veterans_df["Enrollments Project Start Date"], errors="coerce"
).copy()

veterans_df["Enrollments Project Exit Date"] = pd.to_datetime(
    veterans_df["Enrollments Project Exit Date"], errors="coerce"
).copy()

veterans_df["Client Assessments Assessment Date"] = pd.to_datetime(
    veterans_df["Client Assessments Assessment Date"], errors="coerce"
).copy()

veterans_df["Referrals Created Date_completed"] = pd.to_datetime(
    veterans_df["Referrals Created Date_completed"], errors="coerce"
).copy()

veterans_df["Referrals Created Date_pending"] = pd.to_datetime(
    veterans_df["Referrals Created Date_pending"], errors="coerce"
).copy()

Count of veterans in HMIS (row 2)

In [8]:
# Create a new DataFrame to store the results
result_veterans_df = pd.DataFrame(columns=["Month", "Self Identified Vets in HMIS"])

# Generate the last date of each month for the past 12 months
for i in range(1, 13):
    last_date_of_month = (now - pd.DateOffset(months=i)).replace(day=1) + pd.DateOffset(
        months=1, days=-1
    )

    self_identified_vets = veterans_df[
        (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
        & (
            (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
            | (veterans_df["Enrollments Project Exit Date"].isna())
        )
        & (veterans_df["Programs Project Type Code"] != "Homelessness Prevention")
    ].copy()
    # Export the DataFrame to an Excel file
    output_path = os.path.join(
        main_directory_path,
        "extras",
        "calculated_values_drilldown",
        f"{last_date_of_month.strftime('%Y_%m')}",
        f"{last_date_of_month.strftime('%Y_%m')}_self_identified_vets.xlsx",
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    self_identified_vets.to_excel(output_path, index=False, engine="openpyxl")

    count = self_identified_vets["Clients Client ID"].nunique()

    result_veterans_df = pd.concat(
        [
            result_veterans_df,
            pd.DataFrame(
                {
                    "Month": [last_date_of_month.strftime("%Y-%m")],
                    "Self Identified Vets in HMIS": [count],
                }
            ),
        ],
        ignore_index=True,
    )

result_veterans_df.set_index("Month", inplace=True)

Count of Coordinated Entry Vets (Row 3)

In [9]:
# Create a new DataFrame to store the results
result_veterans_df["Coordinated Entry Vets in HMIS"] = 0

# Generate the last date of each month for the past 12 months
for i in range(1, 13):
    last_date_of_month = (now - pd.DateOffset(months=i)).replace(day=1) + pd.DateOffset(
        months=1, days=-1
    )

    coordinated_entry_vets = veterans_df[
        (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
        & (
            (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
            | (veterans_df["Enrollments Project Exit Date"].isna())
        )
        & (veterans_df["Programs Project Type Code"] == "Coordinated Entry")
    ]
    # Export the DataFrame to an Excel file
    output_path = os.path.join(
        main_directory_path,
        "extras",
        "calculated_values_drilldown",
        f"{last_date_of_month.strftime('%Y_%m')}",
        f"{last_date_of_month.strftime('%Y_%m')}_coordinated_entry_vets.xlsx",
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    coordinated_entry_vets.to_excel(output_path, index=False)

    count_coordinated_entry = coordinated_entry_vets["Clients Client ID"].nunique()

    result_veterans_df.loc[
        last_date_of_month.strftime("%Y-%m"), "Coordinated Entry Vets in HMIS"
    ] = count_coordinated_entry

Count of vets on BNL (row 5)

In [10]:
# Create a new column to store the results
result_veterans_df["Vets on BNL"] = 0

# Generate the last date of each month for the past 12 months
for i in range(1, 13):
    last_date_of_month = (now - pd.DateOffset(months=i)).replace(day=1) + pd.DateOffset(
        months=1, days=-1
    )
    one_year_ago = last_date_of_month - pd.DateOffset(years=1)

    bnl_vets = veterans_df[
        (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
        & (
            (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
            | (veterans_df["Enrollments Project Exit Date"].isna())
        )
        & (veterans_df["Programs Project Type Code"] == "Coordinated Entry")
        & (veterans_df["Client Assessments Assessment Date"] >= one_year_ago)
        & (veterans_df["Client Assessments Assessment Date"] <= last_date_of_month)
    ]
    # Export the DataFrame to an Excel file
    output_path = os.path.join(
        main_directory_path,
        "extras",
        "calculated_values_drilldown",
        f"{last_date_of_month.strftime('%Y_%m')}",
        f"{last_date_of_month.strftime('%Y_%m')}_bnl_list_vets.xlsx",
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    bnl_vets.to_excel(output_path, index=False)

    count_vets_on_bnl = bnl_vets["Clients Client ID"].nunique()

    result_veterans_df.loc[last_date_of_month.strftime("%Y-%m"), "Vets on BNL"] = (
        count_vets_on_bnl
    )

In [11]:
# Define the list of eligible discharge statuses
eligible_discharges = [
    "General",
    "Honorable",
    "Eligible: National Guard/Reserves",
    "Ineligible VHA due to length of active service",
    "Other Than Honorable (OTH)",
    "Bad Conduct Discharge (BCD)",
]

# Create a new column to store the results
result_veterans_df["Vets on BNL with Eligible Discharge"] = 0

# Generate the last date of each month for the past 12 months
for i in range(1, 13):
    last_date_of_month = (now - pd.DateOffset(months=i)).replace(day=1) + pd.DateOffset(
        months=1, days=-1
    )
    one_year_ago = last_date_of_month - pd.DateOffset(years=1)

    vets_on_bnl_eligible = veterans_df[
        (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
        & (
            (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
            | (veterans_df["Enrollments Project Exit Date"].isna())
        )
        & (veterans_df["Programs Project Type Code"] == "Coordinated Entry")
        & (veterans_df["Client Assessments Assessment Date"] >= one_year_ago)
        & (veterans_df["Client Assessments Assessment Date"] <= last_date_of_month)
        & (
            veterans_df["Client Assessment Custom Veteran Discharge Status"].isin(
                eligible_discharges
            )
        )
    ]

    # Export the DataFrame to an Excel file
    output_path = os.path.join(
        main_directory_path,
        "extras",
        "calculated_values_drilldown",
        f"{last_date_of_month.strftime('%Y_%m')}",
        f"{last_date_of_month.strftime('%Y_%m')}_bnl_list_vets_eligible.xlsx",
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    vets_on_bnl_eligible.to_excel(output_path, index=False)

    count_vets_on_bnl_eligible = vets_on_bnl_eligible["Clients Client ID"].nunique()

    result_veterans_df.loc[
        last_date_of_month.strftime("%Y-%m"), "Vets on BNL with Eligible Discharge"
    ] = count_vets_on_bnl_eligible

come back to this once referrals is better understood

In [13]:
# Create a new column to store the results
result_veterans_df["Vets not on Community Queue or Referred"] = 0

# Generate the last date of each month for the past 12 months
for i in range(1, 13):
    last_date_of_month = (now - pd.DateOffset(months=i)).replace(day=1) + pd.DateOffset(
        months=1, days=-1
    )
    one_year_ago = last_date_of_month - pd.DateOffset(years=1)

    count_not_on_queue = veterans_df[
        (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
        & (
            (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
            | (veterans_df["Enrollments Project Exit Date"].isna())
        )
        & (veterans_df["Programs Project Type Code"] == "Coordinated Entry")
        & (veterans_df["Client Assessments Assessment Date"] >= one_year_ago)
        & (veterans_df["Client Assessments Assessment Date"] <= last_date_of_month)
        & (
            veterans_df["Client Assessment Custom Veteran Discharge Status"].isin(
                eligible_discharges
            )
        )
        & (veterans_df["Client Referral Status Is client currently on queue?"] == "No")
        & (veterans_df["Referrals Added Date"] < last_date_of_month)
        & (~veterans_df["Referrals Status"].isin(["Pending", "Pending - In Process"]))
    ]["Clients Client ID"].nunique()

    result_veterans_df.loc[
        last_date_of_month.strftime("%Y-%m"), "Vets not on Community Queue or Referred"
    ] = count_not_on_queue

In [12]:
# Create a new column to store the results
result_veterans_df["Vets with DD214"] = 0

# Generate the last date of each month for the past 12 months
for i in range(1, 13):
    last_date_of_month = (now - pd.DateOffset(months=i)).replace(day=1) + pd.DateOffset(
        months=1, days=-1
    )
    one_year_ago = last_date_of_month - pd.DateOffset(years=1)

    vets_with_dd214 = veterans_df[
        (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
        & (
            (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
            | (veterans_df["Enrollments Project Exit Date"].isna())
        )
        & (veterans_df["Programs Project Type Code"] == "Coordinated Entry")
        & (veterans_df["Client Assessments Assessment Date"] >= one_year_ago)
        & (veterans_df["Client Assessments Assessment Date"] <= last_date_of_month)
        & (
            veterans_df["Client Assessment Custom Veteran Discharge Status"].isin(
                eligible_discharges
            )
        )
        & (veterans_df["Clients Client File Name"] == "DD 214 (Veterans Only)")
    ]

    # Export the DataFrame to an Excel file
    output_path = os.path.join(
        main_directory_path,
        "extras",
        "calculated_values_drilldown",
        f"{last_date_of_month.strftime('%Y_%m')}",
        f"{last_date_of_month.strftime('%Y_%m')}_vets_with_dd214.xlsx",
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    vets_with_dd214.to_excel(output_path, index=False)

    count_with_dd214 = vets_with_dd214["Clients Client ID"].nunique()

    result_veterans_df.loc[last_date_of_month.strftime("%Y-%m"), "Vets with DD214"] = (
        count_with_dd214
    )

In [13]:
# Create a new column to store the percentage of vets with DD214
result_veterans_df["Vets with DD214 Percentage"] = (
    (
        result_veterans_df["Vets with DD214"]
        / result_veterans_df["Vets on BNL with Eligible Discharge"]
    )
    * 100
).round(2)

In [14]:
sheltered_programs = [
    "Safe Haven",
    "Transitional Housing",
    "Emergency Shelter – Entry Exit",
]

# Create a new column to store the results
result_veterans_df["Chronic Vets"] = 0
result_veterans_df["Sheltered Vets"] = 0
result_veterans_df["Chronic Sheltered Vets"] = 0

# Generate the last date of each month for the past 12 months
for i in range(1, 13):
    last_date_of_month = (now - pd.DateOffset(months=i)).replace(day=1) + pd.DateOffset(
        months=1, days=-1
    )
    one_year_ago = last_date_of_month - pd.DateOffset(years=1)

    clients_in_transitional_housing = veterans_df[
        (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
        & (
            (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
            | (veterans_df["Enrollments Project Exit Date"].isna())
        )
        & (veterans_df["Programs Project Type Code"] == "Transitional Housing")
    ]

    BNL_eligible_df = veterans_df[
        (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
        & (
            (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
            | (veterans_df["Enrollments Project Exit Date"].isna())
        )
        & (veterans_df["Programs Project Type Code"] == "Coordinated Entry")
        & (veterans_df["Client Assessments Assessment Date"] >= one_year_ago)
        & (veterans_df["Client Assessments Assessment Date"] <= last_date_of_month)
        & (
            veterans_df["Client Assessment Custom Veteran Discharge Status"].isin(
                eligible_discharges
            )
        )
    ].copy()

    BNL_eligible_df.loc[:, "Chronic?"] = BNL_eligible_df.apply(
        lambda row: (
            "Yes"
            if (
                row["Entry Screen Chronically Homeless at Project Start - Individual"]
                == "Yes"
                or (
                    row["Entry Screen Disabling Condition"] == "Yes"
                    and (
                        pd.to_datetime(row["Enrollments Project Start Date"])
                        - pd.to_datetime(
                            row[
                                "Entry Screen Approximate Date this Episode of Homelessness Started Date"
                            ]
                        )
                    ).days
                    >= 365
                )
            )
            and row["Clients Client ID"]
            not in clients_in_transitional_housing["Clients Client ID"].values
            else "No"
        ),
        axis=1,
    )

    chronic_vets = BNL_eligible_df[BNL_eligible_df["Chronic?"] == "Yes"]

    # export chronic vets to excel
    output_path = os.path.join(
        main_directory_path,
        "extras",
        "calculated_values_drilldown",
        f"{last_date_of_month.strftime('%Y_%m')}",
        f"{last_date_of_month.strftime('%Y_%m')}_chronic_vets.xlsx",
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    chronic_vets.to_excel(output_path, index=False)

    count_chronic = BNL_eligible_df[BNL_eligible_df["Chronic?"] == "Yes"][
        "Clients Client ID"
    ].nunique()

    sheltered_vets = veterans_df[
        (veterans_df["Clients Client ID"].isin(BNL_eligible_df["Clients Client ID"]))
        & (veterans_df["Programs Project Type Code"].isin(sheltered_programs))
        & (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
        & (
            (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
            | (veterans_df["Enrollments Project Exit Date"].isna())
        )
    ]

    # export sheltered vets to excel
    output_path = os.path.join(
        main_directory_path,
        "extras",
        "calculated_values_drilldown",
        f"{last_date_of_month.strftime('%Y_%m')}",
        f"{last_date_of_month.strftime('%Y_%m')}_sheltered_vets.xlsx",
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    sheltered_vets.to_excel(output_path, index=False)

    count_sheltered = sheltered_vets["Clients Client ID"].nunique()

    count_chronic_sheltered = veterans_df[
        (
            veterans_df["Clients Client ID"].isin(
                BNL_eligible_df[BNL_eligible_df["Chronic?"] == "Yes"][
                    "Clients Client ID"
                ]
            )
        )
        & (veterans_df["Programs Project Type Code"].isin(sheltered_programs))
        & (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
        & (
            (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
            | (veterans_df["Enrollments Project Exit Date"].isna())
        )
    ]["Clients Client ID"].nunique()

    result_veterans_df.loc[last_date_of_month.strftime("%Y-%m"), "Chronic Vets"] = (
        count_chronic
    )
    result_veterans_df.loc[last_date_of_month.strftime("%Y-%m"), "Sheltered Vets"] = (
        count_sheltered
    )
    result_veterans_df.loc[
        last_date_of_month.strftime("%Y-%m"), "Chronic Sheltered Vets"
    ] = count_chronic_sheltered

In [15]:
result_veterans_df["Unsheltered Vets"] = (
    result_veterans_df["Vets on BNL with Eligible Discharge"]
    - result_veterans_df["Sheltered Vets"]
)

result_veterans_df["Chronic Unsheltered Vets"] = (
    result_veterans_df["Chronic Vets"] - result_veterans_df["Chronic Sheltered Vets"]
)

In [16]:
# Create a new column to store the results
result_veterans_df["Total Outflow"] = 0
result_veterans_df["Total Inflow"] = 0

# Generate the last date of each month for the past 12 months
for i in range(1, 13):
    last_date_of_month = (now - pd.DateOffset(months=i)).replace(day=1) + pd.DateOffset(
        months=1, days=-1
    )

    previous_month = (now - pd.DateOffset(months=i + 1)).replace(day=1) + pd.DateOffset(
        months=1, days=-1
    )
    one_year_ago = last_date_of_month - pd.DateOffset(years=1)
    previous_one_year_ago = previous_month - pd.DateOffset(years=1)
    previous_month_bnl = veterans_df[
        (veterans_df["Enrollments Project Start Date"] <= previous_month)
        & (
            (veterans_df["Enrollments Project Exit Date"] > previous_month)
            | (veterans_df["Enrollments Project Exit Date"].isna())
        )
        & (veterans_df["Programs Project Type Code"] == "Coordinated Entry")
        & (veterans_df["Client Assessments Assessment Date"] >= previous_one_year_ago)
        & (veterans_df["Client Assessments Assessment Date"] <= previous_month)
        & (
            veterans_df["Client Assessment Custom Veteran Discharge Status"].isin(
                eligible_discharges
            )
        )
    ]
    current_month_bnl = veterans_df[
        (veterans_df["Enrollments Project Start Date"] <= last_date_of_month)
        & (
            (veterans_df["Enrollments Project Exit Date"] > last_date_of_month)
            | (veterans_df["Enrollments Project Exit Date"].isna())
        )
        & (veterans_df["Programs Project Type Code"] == "Coordinated Entry")
        & (veterans_df["Client Assessments Assessment Date"] >= one_year_ago)
        & (veterans_df["Client Assessments Assessment Date"] <= last_date_of_month)
        & (
            veterans_df["Client Assessment Custom Veteran Discharge Status"].isin(
                eligible_discharges
            )
        )
    ]

    outflow_vets_bnl = previous_month_bnl[
        ~previous_month_bnl["Clients Client ID"].isin(
            current_month_bnl["Clients Client ID"]
        )
    ]

    # export outflow vets to excel
    output_path = os.path.join(
        main_directory_path,
        "extras",
        "calculated_values_drilldown",
        f"{last_date_of_month.strftime('%Y_%m')}",
        f"{last_date_of_month.strftime('%Y_%m')}_outflow_vets.xlsx",
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    outflow_vets_bnl.to_excel(output_path, index=False)

    count_outflow_bnl = outflow_vets_bnl["Clients Client ID"].nunique()

    inflow_vets_bnl = current_month_bnl[
        ~current_month_bnl["Clients Client ID"].isin(
            previous_month_bnl["Clients Client ID"]
        )
    ]

    # export inflow vets to excel
    output_path = os.path.join(
        main_directory_path,
        "extras",
        "calculated_values_drilldown",
        f"{last_date_of_month.strftime('%Y_%m')}",
        f"{last_date_of_month.strftime('%Y_%m')}_inflow_vets.xlsx",
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    inflow_vets_bnl.to_excel(output_path, index=False)

    count_inflow_bnl = inflow_vets_bnl["Clients Client ID"].nunique()

    result_veterans_df.loc[last_date_of_month.strftime("%Y-%m"), "Total Outflow"] = (
        count_outflow_bnl
    )

    result_veterans_df.loc[last_date_of_month.strftime("%Y-%m"), "Total Inflow"] = (
        count_inflow_bnl
    )

In [17]:
result_veterans_df

,Self Identified Vets in HMIS,Coordinated Entry Vets in HMIS,Vets on BNL,Vets on BNL with Eligible Discharge,Vets with DD214,Vets with DD214 Percentage,Chronic Vets,Sheltered Vets,Chronic Sheltered Vets,Unsheltered Vets,Chronic Unsheltered Vets,Total Outflow,Total Inflow
Month,,,,,,,,,,,,,
2024-12,667,401,255,215,183,85.12,55,98,13,117,42,17,16
2024-11,655,393,249,216,182,84.26,58,100,11,116,47,16,24
2024-10,649,380,238,208,180,86.54,55,96,10,112,45,20,29
2024-09,628,363,228,199,172,86.43,56,91,11,108,45,18,22
2024-08,623,356,218,195,168,86.15,57,86,12,109,45,28,19
2024-07,621,359,225,204,173,84.80,54,92,12,112,42,19,19
2024-06,626,356,227,204,172,84.31,56,89,11,115,45,20,26
2024-05,613,349,219,198,170,85.86,50,86,10,112,40,33,28
2024-04,595,342,223,203,168,82.76,57,85,9,118,48,15,28


In [63]:
result_veterans_df.reset_index(inplace=True)

In [64]:
result_veterans_df["Update Time HST"] = now_hst_str
result_veterans_df["Update Time CST"] = now_str_formatted
result_veterans_df["Update Time UTC"] = now_utc_str
conn = GsAutomationReliefAssistanceConnection(sql_username, sql_password, auth=True)
conn.insert_with_progress(
    df=result_veterans_df,
    schema="dbo",
    tbl_nm="partners_in_care_veterans_summary",
    dtypes={},
)

Building connection to relief-assistance.database.windows.net - gs-automation


10000it [00:07, 1300.81it/s]          


In [65]:
logger.cleanup()

In [66]:
if task_mgr._process_success != "Success":
    subject, body = task_mgr.generate_task_email(today_str)

    send_task_alert(subject, body)